# 📊 Tech-Enabled America Fund — Portfolio Analytics
## Notebook 1 · Data & Returns

| | |
|---|---|
| **Fund** | Tech-Enabled America Fund Ltd. |
| **Period** | January 2017 – December 2022 |
| **Universe** | 11 US stocks · all 11 GICS sectors |
| **Construction** | Mean-variance optimisation (min/max weight constraints) |
| **Data source** | `FINC71-303_ExcelSheet_14047473_LorenaBrant.xlsx` |

---
**Run order:** `01` → `02` → `03` → `04`

In [ ]:
# Install once: pip install -r requirements.txt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (13, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
    'axes.titlesize': 13,
})
BLUE, RED, GREEN, GOLD = '#1a5276', '#c0392b', '#1e8449', '#d4ac0d'

# Load all data
port_ret   = pd.read_csv('../data/portfolio_monthly_returns.csv', index_col=0, parse_dates=True).squeeze()
stock_ret  = pd.read_csv('../data/stock_monthly_returns.csv',    index_col=0, parse_dates=True)
annual_ret = pd.read_csv('../data/annual_returns.csv',           index_col=0)
ann_stocks = pd.read_csv('../data/annual_stock_returns.csv')
stock_cagr = pd.read_csv('../data/stock_cagr.csv')

with open('../data/portfolio_config.json') as f:
    PORTFOLIO = json.load(f)
with open('../data/confirmed_metrics.json') as f:
    METRICS = json.load(f)

TICKERS = list(PORTFOLIO.keys())
WEIGHTS = np.array([PORTFOLIO[t]['weight'] for t in TICKERS])
RF      = METRICS['RF']

print(f"Portfolio:  {len(TICKERS)} stocks  |  RF: {RF*100:.2f}%")
print(f"Returns:    {len(port_ret)} months  |  {port_ret.index[0].strftime('%b %Y')} → {port_ret.index[-1].strftime('%b %Y')}")

## 1 · Portfolio Holdings

In [ ]:
port_df = pd.DataFrame(PORTFOLIO).T.reset_index().rename(columns={'index': 'Ticker'})
port_df['Weight'] = (port_df['weight'].astype(float) * 100).round(2).astype(str) + '%'
display(port_df[['Ticker', 'name', 'sector', 'Weight']].rename(
    columns={'name': 'Company', 'sector': 'GICS Sector'}
).set_index('Ticker'))

# Pie chart of sector weights
fig, ax = plt.subplots(figsize=(9, 5))
sectors = [(PORTFOLIO[t]['sector'], PORTFOLIO[t]['weight']) for t in TICKERS]
sector_df = pd.DataFrame(sectors, columns=['Sector', 'Weight'])
sector_df = sector_df.groupby('Sector')['Weight'].sum().sort_values(ascending=False)
colors = plt.cm.Blues(np.linspace(0.35, 0.85, len(sector_df)))
wedges, texts, autotexts = ax.pie(
    sector_df.values, labels=sector_df.index, autopct='%1.1f%%',
    colors=colors, startangle=90, pctdistance=0.8
)
for t in texts: t.set_fontsize(9)
for a in autotexts: a.set_fontsize(8)
ax.set_title('Portfolio Allocation by GICS Sector', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/sector_allocation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2 · Annual Returns vs Benchmarks

In [ ]:
print("Annual Returns (%)")
display((annual_ret * 100).round(2))

fig, ax = plt.subplots(figsize=(13, 5))
years = [int(c) for c in annual_ret.columns]
x, w = np.arange(len(years)), 0.26
palette = {'Portfolio': BLUE, 'S&P 500': RED, 'NASDAQ': GREEN}

for i, (series, color) in enumerate(palette.items()):
    vals = annual_ret.loc[series].values * 100
    bars = ax.bar(x + i*w, vals, width=w, label=series,
                  color=color, alpha=0.88, edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ypos = bar.get_height() + 0.8 if v >= 0 else bar.get_height() - 2.2
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f'{v:.1f}%', ha='center', fontsize=8, fontweight='bold', color='#333')

ax.set_xticks(x + w); ax.set_xticklabels(years, fontsize=11)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Annual Return (%)')
ax.set_title('Annual Returns: Tech-Enabled America Fund vs Benchmarks (2017–2022)', fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.savefig('../data/annual_returns.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 · Cumulative Wealth ($100m)

In [ ]:
# Build smooth monthly cumulative series from confirmed annual returns
def ann_to_monthly_cum(annual_dict, start=100):
    vals, current = {}, start
    for yr, ann in sorted(annual_dict.items()):
        r_m = (1 + ann) ** (1/12) - 1
        for m in range(1, 13):
            current *= (1 + r_m)
            vals[pd.Timestamp(f'{yr}-{m:02d}-01')] = current
    return pd.Series(vals)

cum_port = (1 + port_ret).cumprod() * 100
cum_sp5  = ann_to_monthly_cum({int(k): v for k, v in annual_ret.loc['S&P 500'].items()})
cum_nas  = ann_to_monthly_cum({int(k): v for k, v in annual_ret.loc['NASDAQ'].items()})

fig, ax = plt.subplots(figsize=(13, 6))
lw = 2.3
ax.plot(cum_port.index, cum_port, color=BLUE,  lw=lw,   label='Portfolio (Tech-Enabled America Fund)')
ax.plot(cum_sp5.index,  cum_sp5,  color=RED,   lw=lw-0.5, label='S&P 500',   linestyle='--')
ax.plot(cum_nas.index,  cum_nas,  color=GREEN, lw=lw-0.5, label='NASDAQ-100', linestyle=':')
ax.axhline(100, color='gray', lw=0.7, linestyle='-', alpha=0.4)

# End-value labels
for label, s, c in [('$338m', cum_port, BLUE), ('$171m', cum_sp5, RED), ('$225m', cum_nas, GREEN)]:
    ax.annotate(label, xy=(s.index[-1], s.iloc[-1]),
                xytext=(10, 0), textcoords='offset points',
                fontsize=11, color=c, fontweight='bold')

# Macro events
events = {'2018-10-01': 'Trade War', '2020-03-01': 'COVID-19', '2022-01-01': 'Rate Hikes'}
for dt_str, lbl in events.items():
    dt = pd.Timestamp(dt_str)
    ax.axvline(dt, color='gray', lw=0.8, linestyle=':', alpha=0.7)
    ax.text(dt, 50, lbl, fontsize=8.5, color='#555', ha='center', rotation=90, va='bottom')

ax.set_ylabel('Portfolio Value ($m)')
ax.set_title('Cumulative Performance: $100m Initial Investment (Jan 2017 – Dec 2022)', fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}m'))
plt.tight_layout()
plt.savefig('../data/cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
summary = pd.DataFrame({
    'Final Value': ['$338m', '$225m', '$171m'],
    'CAGR':        ['22.50%', '14.45%', '9.41%'],
}, index=['Portfolio', 'NASDAQ', 'S&P 500'])
print(summary.to_string())

## 4 · Stock-Level CAGR: Top & Bottom Performers

In [ ]:
cagr_df = stock_cagr.sort_values('CAGR').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 5))
bar_colors = [RED if v < 0 else (GOLD if v < 15 else GREEN) for v in cagr_df['CAGR']]
bars = ax.barh(cagr_df['Ticker'], cagr_df['CAGR'], color=bar_colors, edgecolor='white', alpha=0.88)

for bar, v in zip(bars, cagr_df['CAGR']):
    pad = 0.4 if v >= 0 else -0.4
    ax.text(v + pad, bar.get_y() + bar.get_height()/2,
            f'{v:.2f}%', va='center', ha='left' if v >= 0 else 'right',
            fontsize=10, fontweight='bold')

ax.axvline(0, color='black', lw=0.9)
ax.set_xlabel('CAGR (%) 2017–2022')
ax.set_title('Stock CAGR: Top & Bottom Performers (2017–2022)', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.savefig('../data/stock_cagr_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key insight: NVDA (+32.58% p.a.) drove fund outperformance.")
print("SLB (-4.38% p.a.) was the drag — energy services hit by capex cuts and trade war.")

## 5 · Annual Returns Heatmap by Stock

In [ ]:
heat = ann_stocks.set_index('Ticker')[[c for c in ann_stocks.columns if c.isdigit() or (len(c)==4 and c[:2] in ('20','19'))]]
# Get year columns
year_cols = [str(y) for y in range(2017, 2023)]
heat = ann_stocks.set_index('Ticker')[year_cols].astype(float) * 100

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(
    heat, annot=True, fmt='.1f', center=0,
    cmap='RdYlGn', linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Annual Return (%)', 'shrink': 0.8}
)
ax.set_title('Annual Returns by Stock — 2017 to 2022', fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('Year')
plt.tight_layout()
plt.savefig('../data/annual_stock_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNotebook 01 complete ✓')